In [5]:
# 1. Install the specific 'prebuilt' support package
%pip install -U langgraph-prebuilt

# 2. Re-sync the main packages to ensure they match
%pip install -U langchain langgraph langchain-groq

  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.6
    Uninstalling langgraph-prebuilt-1.0.6:
      Successfully uninstalled langgraph-prebuilt-1.0.6
Note: you may need to restart the kernel to use updated packages.

  Attempting uninstall: langgraph

    Found existing installation: langgraph 1.0.6

    Uninstalling langgraph-1.0.6:

      Successfully uninstalled langgraph-1.0.6

   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
   ---------------------------------------- 0/2 [langgraph]
  Attempting uninstall: langchain
   ---------------------------------------- 0/2 [langgraph]
    Found existing installation: langchain 1.2.6
   ---------------------------------------- 0/2 [langgraph]
    Uninstalling langchain-1.2.6:
   -------------------------

In [6]:
%pip show langgraph.prebuilt.tool_node

Note: you may need to restart the kernel to use updated packages.


In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile" ,
    groq_api_key=os.getenv("OPENAI_API_KEY") ,
    temperature=0 ,
)

In [9]:
from langchain_core.tools import tool

@tool
def calculate_travel_time(distance_km: float, speed_kmh: float) -> str :
    """Calculate travel time given distance and speed."""
    
    time = distance_km / speed_kmh
    return f"The estimate travel time is {time:.2f} hours."

tools = [calculate_travel_time]

In [10]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Use your tools for calculations."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful travel assistant. Use your tools for calculations."
)

# Test the agent
result = agent.invoke({
    "messages": [("human", "How long will it take to drive 500km at 100km/h?")]
})

print("\n--- Final Output ---")
print(result["messages"][-1].content)


--- Final Output ---
The estimated travel time is 5 hours.


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from pydantic import BaseModel

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

class TravelInfo(BaseModel) :
    distance: float
    speed: float
    time: float
    message: str

google_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful travel assistant. Use your tools for calculations. ALWAYS provide your final answer using the TravelInfo format.",
    response_format= TravelInfo
)

# Test the agent
# for token, metadata in agent.stream(
#     {
#     "messages": [
#             HumanMessage(content = "How long will it take to drive 500km at 100km/h?"),
#             AIMessage(content = "The estimated travel time is 5 hours."),
#             HumanMessage(content = "Which vehical is best for this travel?")
#         ]
#     },
#     stream_mode="messages"
# ) :
#     if token.content:  # Check if there's actual content
#         print(token.content, end="", flush=True) 

result = google_agent.invoke(
    {"messages": [HumanMessage(content = "How long will it take to drive 500km at 100km/h?")]}
)

print("\n--- Final Output ---")
print(result)
print(result["messages"][-1].content)


--- Final Output ---
{'messages': [HumanMessage(content='How long will it take to drive 500km at 100km/h?', additional_kwargs={}, response_metadata={}, id='c7a644a2-fb51-4146-a966-7a8f8293c7a4'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'calculate_travel_time', 'arguments': '{"distance_km": 500, "speed_kmh": 100}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019beeb0-338a-7013-a2eb-d74add119690-0', tool_calls=[{'name': 'calculate_travel_time', 'args': {'distance_km': 500, 'speed_kmh': 100}, 'id': 'f4cbb1a4-c943-4f56-bd91-a35d9aa7ed07', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 173, 'output_tokens': 31, 'total_tokens': 204, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='The estimate travel time is 5.00 hours.', name='calculate_travel_time', id='b3c8f10d-07eb-42ab-be14-e9cbb0cb965c', tool_call_id='f4

In [12]:
print(result["structured_response"])

distance=500.0 speed=100.0 time=5.0 message='The estimate travel time is 5.00 hours.'


In [13]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The current mayor is Democrat Daniel Lurie.',
   'score': 0.95324475,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly approach.',
   'score': 0.8755427,
   'raw_content': None},
  {'url': 'https://www.sf.gov/departments--office-mayor',
   'title': 'Office of the Mayor - SF.gov',
   'content': 'Daniel Lurie is the 46th Mayor of the City and County of San Francisco.',
   'score': 0.8488849,
   'raw_content': None

In [14]:
agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite"),
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='42511b0d-2a85-4cda-abf6-cd9b88a5b86d'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "current mayor of San Francisco"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019beeb0-47d0-7441-bb66-621c2983fd58-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco'}, 'id': 'e9735dcf-f7ac-40dd-a636-4968d068e562', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 19, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"query": "current mayor of San Francisco", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://en.wikipedia.org/wiki/Mayor_of_San_Francisco", "titl

## Memory

In [19]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from pprint import pprint

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite"),
    checkpointer=InMemorySaver()
)

In [20]:
q1 = HumanMessage(content="Give me today's latest news.")
q2 = HumanMessage(content="Give me today's latest news of Bangladesh.")
config = {"configurable": {"thread_id": "1"}}

In [21]:
response = agent.invoke(
    {"messages": [q1]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content="Give me today's latest news.", additional_kwargs={}, response_metadata={}, id='470b76eb-2a65-488f-8fc7-9a0b14f46622'),
              AIMessage(content='As an AI, I don\'t have real-time access to breaking news feeds like a human would. My knowledge is based on the data I was trained on, which has a cutoff point. Therefore, I can\'t give you "today\'s latest news" in the way a news website or broadcast would.\n\n**To get today\'s latest news, I recommend checking these reliable sources:**\n\n*   **Major News Websites:**\n    *   Associated Press (AP)\n    *   Reuters\n    *   BBC News\n    *   CNN\n    *   The New York Times\n    *   The Wall Street Journal\n    *   Your local news outlets\n\n*   **News Apps:** Many of the above also have dedicated apps that provide real-time updates.\n\n*   **Social Media (with caution):** While news organizations often post updates on social media, it\'s best to verify information from official sources.\n\n**If you 

In [22]:
response = agent.invoke(
    {"messages": [q2]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content="Give me today's latest news.", additional_kwargs={}, response_metadata={}, id='470b76eb-2a65-488f-8fc7-9a0b14f46622'),
              AIMessage(content='As an AI, I don\'t have real-time access to breaking news feeds like a human would. My knowledge is based on the data I was trained on, which has a cutoff point. Therefore, I can\'t give you "today\'s latest news" in the way a news website or broadcast would.\n\n**To get today\'s latest news, I recommend checking these reliable sources:**\n\n*   **Major News Websites:**\n    *   Associated Press (AP)\n    *   Reuters\n    *   BBC News\n    *   CNN\n    *   The New York Times\n    *   The Wall Street Journal\n    *   Your local news outlets\n\n*   **News Apps:** Many of the above also have dedicated apps that provide real-time updates.\n\n*   **Social Media (with caution):** While news organizations often post updates on social media, it\'s best to verify information from official sources.\n\n**If you 

## Personal Assistant

In [40]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from tavily import TavilyClient
from langchain.tools import tool
from typing import Dict, Any
from pprint import pprint
import os

In [41]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
tavily_client = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))
config = {"configurable": {"thread_id": "1"}}

In [42]:
system_prompt = '''
## ROLE
You are the "Personal Physics Master," an elite computational physicist and tutor. Your mission is to solve complex physics problems with absolute mathematical precision while providing intuitive, conceptual clarity.

## CAPABILITIES & WORKFLOW
1.  **Analyze**: Carefully decompose the user's problem into known variables, unknown targets, and relevant physical laws (Newtonian, Thermodynamic, Quantum, etc.).
2.  **Reason First**: Always "Think Step-by-Step" before taking action. State the principles involved (e.g., "Conservation of Energy") before doing math.
3.  **Tool Integration**:
    * Use **Web Search** for real-world constants (e.g., the density of titanium), historical context, or specific formulas not in your training data.
    * Use **Computational Tools** (like a Python REPL or Calculator) for all multi-step math and unit conversions to avoid "hallucinated" arithmetic errors.
4.  **Self-Correction**: If a result seems physically impossible (e.g., a speed faster than light or a negative Kelvin temperature), acknowledge it, re-search for the correct parameters, and solve again.

## GUIDELINES
- **Units Matter**: Always include units in your steps (m/s, Joules, etc.) and perform dimensional analysis to verify the solution.
- **Tone**: Professional, encouraging, and intellectually rigorous—like a senior professor.
- **Formatting**: Use LaTeX for all mathematical expressions and standalone equations.
- **Hybrid Solving**: You are encouraged to combine your internal knowledge with web-verified data to provide the most robust answer possible.

## CONSTRAINTS
- Never provide a "final answer" without showing the derivation.
- If a problem is underspecified, ask for the missing parameters before searching or solving.
'''

In [43]:
@tool
def calculate_travel_time(distance_km: float, speed_kmh: float) -> str :
    """Calculate travel time given distance and speed."""
    
    time = distance_km / speed_kmh
    return f"The estimate travel time is {time:.2f} hours."

@tool
def calculate_travel_speed(distance_km: float, time: float) -> str :
    """Calculate travel speed given distance and time."""

    speed = distance_km / time
    return f"The estimate travel speed is {speed:.2f} km/hr."

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

tools = [calculate_travel_speed, calculate_travel_time, web_search]

In [44]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

In [45]:
while True :
    q_inp = input("User: ")

    if q_inp in ["exit", "quit", "bye"] :
        print("AI: Have a nice day! Bye.")
        break

    print(f"User: {q_inp}")
    question = HumanMessage(content=q_inp)
    print("AI: ", end="")
    for token, metadata in agent.stream(
        {"messages": [question]},
        stream_mode="messages",
        config=config,
    ):
        if token.content:
            print(token.content, end="", flush=True)
    print()

User: hi
AI: Hello! I am the Personal Physics Master. How can I help you solve a physics problem today?
User: what is time
AI: 

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 19.21008837s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flas